# Survived 生存予測モデルの探索と視覚化

このノートブックでは、Survived（タイタニック）データセットを使った二値分類モデルの性能を視覚的に評価します。

## 目的

- **グループ別欠損値補完を実践する** - Pclass と Survived でグループ化
- **カテゴリカル変数のエンコーディング** - Sex を OneHotEncoding
- **二値分類モデルの性能を視覚的に評価する** - 混同行列、ROC 曲線
- **クラス不均衡への対応** - データ分布の理解

## 1. 環境セットアップとパッケージ読み込み

In [ ]:
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.FastTree, 3.0.1"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: FSharp.Stats, 0.5.0"

open System
open System.IO
open Microsoft.ML
open Microsoft.ML.Data
open Plotly.NET
open Plotly.NET.LayoutObjects
open FSharp.Stats

printfn "✅ 環境セットアップ完了"

## 2. データ型定義

In [ ]:
[<CLIMutable>]
type SurvivedData = {
    [<LoadColumn(0)>] PassengerId: int
    [<LoadColumn(1)>] Survived: bool
    [<LoadColumn(2)>] Pclass: float32
    [<LoadColumn(3)>] Sex: string
    [<LoadColumn(4)>] Age: float32
}

[<CLIMutable>]
type SurvivedPrediction = {
    [<ColumnName("PredictedLabel")>] Survived: bool
    Score: float32
    Probability: float32
}

## 3. データ読み込みと探索

In [ ]:
let mlContext = MLContext(seed = Nullable 0)
let dataPath = "../data/Survived.csv"

let dataView =
    mlContext.Data.LoadFromTextFile<SurvivedData>(
        dataPath,
        hasHeader = true,
        separatorChar = ',')

// データフレームに変換
let survivedData =
    mlContext.Data.CreateEnumerable<SurvivedData>(dataView, reuseRowObject = false)
    |> Seq.toList

printfn $"データ数: {survivedData.Length} サンプル"
printfn $"\n最初の 5 件:"
survivedData
|> List.take 5
|> List.iteri (fun i d ->
    let survived = if d.Survived then "生存" else "死亡"
    printfn $"  {i+1}. ID={d.PassengerId}, 生存={survived}, Pclass={int d.Pclass}, Sex={d.Sex}, Age={d.Age:F1}"
)

## 4. クラス分布の確認

In [ ]:
printfn "\n=== クラス分布 ==="
let classCounts =
    survivedData
    |> List.groupBy (fun d -> d.Survived)
    |> List.map (fun (survived, samples) ->
        let label = if survived then "生存" else "死亡"
        let count = samples.Length
        let percentage = float count / float survivedData.Length * 100.0
        printfn $"  {label}: {count} サンプル ({percentage:F2}%%)"
        (label, count))

// クラス分布の可視化
let labels = classCounts |> List.map fst
let values = classCounts |> List.map (snd >> float)

let pieChart =
    Chart.Pie(values, labels)
    |> Chart.withTitle "生存/死亡の分布（クラス不均衡）"
    |> Chart.withSize(600, 500)

pieChart

## 5. 欠損値の確認

In [ ]:
let missingAges =
    survivedData
    |> List.filter (fun d -> Single.IsNaN(d.Age))
    |> List.length

let missingPercentage = float missingAges / float survivedData.Length * 100.0
printfn "\n=== 欠損値の確認 ==="
printfn $"Age の欠損値: {missingAges} 件 ({missingPercentage:F2}%%)"

## 6. 基本統計量

In [ ]:
printfn "\n=== 基本統計量 ==="

printfn "\nAge（年齢）:"
let validAges =
    survivedData
    |> List.map (fun d -> float d.Age)
    |> List.filter (fun v -> not (Double.IsNaN v))

if validAges.Length > 0 then
    let mean = List.average validAges
    let std = Seq.stDev validAges
    let min = List.min validAges
    let max = List.max validAges
    printfn $"  平均={mean:F2}, 標準偏差={std:F2}, 最小={min:F2}, 最大={max:F2}"

printfn "\nPclass（客室クラス）別の統計:"
survivedData
|> List.groupBy (fun d -> int d.Pclass)
|> List.sortBy fst
|> List.iter (fun (pclass, samples) ->
    let survived = samples |> List.filter (fun d -> d.Survived) |> List.length
    let total = samples.Length
    let survivalRate = float survived / float total * 100.0
    printfn $"  クラス {pclass}: {total} 人（生存率 {survivalRate:F2}%%）"
)

printfn "\nSex（性別）別の統計:"
survivedData
|> List.groupBy (fun d -> d.Sex)
|> List.iter (fun (sex, samples) ->
    let survived = samples |> List.filter (fun d -> d.Survived) |> List.length
    let total = samples.Length
    let survivalRate = float survived / float total * 100.0
    printfn $"  {sex}: {total} 人（生存率 {survivalRate:F2}%%）"
)

## 7. データ分布の視覚化

### 年齢の分布

In [ ]:
let ageHist =
    Chart.Histogram(validAges, Name = "年齢分布")
    |> Chart.withXAxisStyle(TitleText = "年齢")
    |> Chart.withYAxisStyle(TitleText = "頻度")
    |> Chart.withTitle "乗客の年齢分布"
    |> Chart.withSize(800, 500)

ageHist

### 客室クラス別の生存率

In [ ]:
let pclassSurvival =
    survivedData
    |> List.groupBy (fun d -> int d.Pclass)
    |> List.sortBy fst
    |> List.map (fun (pclass, samples) ->
        let survived = samples |> List.filter (fun d -> d.Survived) |> List.length
        let total = samples.Length
        (string pclass, float survived / float total * 100.0))

let pclassLabels = pclassSurvival |> List.map fst
let pclassRates = pclassSurvival |> List.map snd

let pclassBarChart =
    Chart.Column(pclassLabels, pclassRates)
    |> Chart.withXAxisStyle(TitleText = "客室クラス")
    |> Chart.withYAxisStyle(TitleText = "生存率 (%)")
    |> Chart.withTitle "客室クラス別の生存率"
    |> Chart.withSize(800, 500)

pclassBarChart

### 性別別の生存率

In [ ]:
let sexSurvival =
    survivedData
    |> List.groupBy (fun d -> d.Sex)
    |> List.map (fun (sex, samples) ->
        let survived = samples |> List.filter (fun d -> d.Survived) |> List.length
        let total = samples.Length
        (sex, float survived / float total * 100.0))

let sexLabels = sexSurvival |> List.map fst
let sexRates = sexSurvival |> List.map snd

let sexBarChart =
    Chart.Column(sexLabels, sexRates)
    |> Chart.withXAxisStyle(TitleText = "性別")
    |> Chart.withYAxisStyle(TitleText = "生存率 (%)")
    |> Chart.withTitle "性別別の生存率"
    |> Chart.withSize(800, 500)

sexBarChart

## 8. グループ別欠損値補完

In [ ]:
// グループ別平均を計算
let calculateGroupMeans (rows: SurvivedData list) =
    rows
    |> List.filter (fun r -> not (Single.IsNaN(r.Age)))
    |> List.groupBy (fun r -> (r.Pclass, r.Survived))
    |> List.map (fun ((pclass, survived), group) ->
        let avgAge = group |> List.averageBy (fun r -> r.Age)
        ((pclass, survived), avgAge))
    |> Map.ofList

let ageMapping = calculateGroupMeans survivedData

printfn "\n=== グループ別平均年齢 ==="
ageMapping
|> Map.toList
|> List.sortBy fst
|> List.iter (fun ((pclass, survived), avgAge) ->
    let label = if survived then "生存" else "死亡"
    printfn $"  クラス {int pclass}, {label}: {avgAge:F2} 歳"
)

// 欠損値補完
let imputedData =
    survivedData
    |> List.map (fun row ->
        if Single.IsNaN(row.Age) then
            let key = (row.Pclass, row.Survived)
            match Map.tryFind key ageMapping with
            | Some avgAge -> { row with Age = avgAge }
            | None -> row
        else
            row
    )

printfn $"\n欠損値補完後のデータ数: {imputedData.Length} サンプル"

## 9. モデルの訓練と評価

In [ ]:
// F# 用のダウンキャストヘルパー関数
let downcastPipeline (x: IEstimator<_>) =
    match x with
    | :? IEstimator<ITransformer> as y -> y
    | _ -> failwith "downcastPipeline: IEstimator<ITransformer> が期待されます"

let imputedDataView = mlContext.Data.LoadFromEnumerable(imputedData)
let trainTestSplit = mlContext.Data.TrainTestSplit(imputedDataView, testFraction = 0.2, seed = Nullable 42)

let pipeline =
    mlContext.Transforms.CopyColumns("Label", "Survived")
    |> downcastPipeline
    |> fun estimator ->
        estimator
            .Append(mlContext.Transforms.Categorical.OneHotEncoding("SexEncoded", "Sex"))
            .Append(mlContext.Transforms.Concatenate("Features", "Pclass", "SexEncoded", "Age"))
            .Append(mlContext.BinaryClassification.Trainers.FastTree())

printfn "\nモデルを訓練中..."
let model = pipeline.Fit(trainTestSplit.TrainSet)

// 予測
let predictions = model.Transform(trainTestSplit.TestSet)

// 評価
let metrics = mlContext.BinaryClassification.Evaluate(predictions, labelColumnName = "Label")

printfn "\n=== モデル評価結果 ==="
printfn $"精度 (Accuracy):        {metrics.Accuracy:F4} ({metrics.Accuracy * 100.0:F2}%%)"
printfn $"AUC:                    {metrics.AreaUnderRocCurve:F4}"
printfn $"F1 スコア:              {metrics.F1Score:F4}"
printfn $"適合率 (Precision):     {metrics.PositivePrecision:F4}"
printfn $"再現率 (Recall):        {metrics.PositiveRecall:F4}"

## 10. 混同行列の可視化

In [ ]:
let testData =
    mlContext.Data.CreateEnumerable<SurvivedData>(trainTestSplit.TestSet, reuseRowObject = false)
    |> Seq.toList

let predictedData =
    mlContext.Data.CreateEnumerable<SurvivedPrediction>(predictions, reuseRowObject = false)
    |> Seq.toList

// 混同行列の計算
let confusionPairs =
    List.zip testData predictedData
    |> List.groupBy (fun (actual, predicted) -> (actual.Survived, predicted.Survived))
    |> List.map (fun ((actualLabel, predLabel), items) -> ((actualLabel, predLabel), items.Length))
    |> Map.ofList

let tp = Map.tryFind (true, true) confusionPairs |> Option.defaultValue 0
let fp = Map.tryFind (false, true) confusionPairs |> Option.defaultValue 0
let tn = Map.tryFind (false, false) confusionPairs |> Option.defaultValue 0
let fn = Map.tryFind (true, false) confusionPairs |> Option.defaultValue 0

printfn "\n=== 混同行列 ==="
printfn "              予測"
printfn "        | 死亡 | 生存"
printfn "--------|------|------"
printfn $"死亡    | {tn,4} | {fp,4}"
printfn $"生存    | {fn,4} | {tp,4}"

// ヒートマップで可視化
let confusionMatrix = [[float tn; float fp]; [float fn; float tp]]
let xLabels = ["死亡（予測）"; "生存（予測）"]
let yLabels = ["死亡（実測）"; "生存（実測）"]

let heatmap =
    Chart.Heatmap(confusionMatrix, X = xLabels, Y = yLabels, ColorScale = StyleParam.Colorscale.Viridis)
    |> Chart.withTitle "混同行列"
    |> Chart.withSize(600, 600)

heatmap

## 11. ROC 曲線の可視化

In [ ]:
// スコアと実際のラベルを取得
let scores = predictedData |> List.map (fun p -> float p.Score)
let actualLabels = testData |> List.map (fun d -> d.Survived)

// 簡易的な ROC 曲線の計算
let sortedData =
    List.zip scores actualLabels
    |> List.sortByDescending fst

let positives = actualLabels |> List.filter id |> List.length |> float
let negatives = actualLabels |> List.filter not |> List.length |> float

let rocPoints =
    sortedData
    |> List.scan (fun (tp, fp) (_, label) ->
        if label then (tp + 1.0, fp)
        else (tp, fp + 1.0)
    ) (0.0, 0.0)
    |> List.map (fun (tp, fp) -> (fp / negatives, tp / positives))

let fpr = rocPoints |> List.map fst
let tpr = rocPoints |> List.map snd

// ROC 曲線
let rocCurve =
    Chart.Line(fpr, tpr, Name = "ROC 曲線")
    |> Chart.withLineStyle(Width = 2)

// ランダム分類器の基準線
let randomLine =
    Chart.Line([0.0; 1.0], [0.0; 1.0], Name = "ランダム分類器")
    |> Chart.withLineStyle(Dash = StyleParam.DrawingStyle.Dash, Color = Color.fromKeyword Gray)

let rocChart =
    [rocCurve; randomLine]
    |> Chart.combine
    |> Chart.withXAxisStyle(TitleText = "偽陽性率 (FPR)")
    |> Chart.withYAxisStyle(TitleText = "真陽性率 (TPR)")
    |> Chart.withTitle $"ROC 曲線 (AUC = {metrics.AreaUnderRocCurve:F4})"
    |> Chart.withSize(800, 600)

rocChart

## 12. 予測例

In [ ]:
printfn "\n=== 予測例 ==="

let testSamples = [
    { PassengerId = 0; Survived = false; Pclass = 3.0f; Sex = "male"; Age = 22.0f }
    { PassengerId = 0; Survived = false; Pclass = 1.0f; Sex = "female"; Age = 38.0f }
    { PassengerId = 0; Survived = false; Pclass = 2.0f; Sex = "male"; Age = 35.0f }
]

testSamples
|> List.iteri (fun i sample ->
    let input = mlContext.Data.LoadFromEnumerable([sample])
    let prediction = model.Transform(input)
    let pred =
        mlContext.Data.CreateEnumerable<SurvivedPrediction>(prediction, reuseRowObject = false)
        |> Seq.head

    let predLabel = if pred.Survived then "生存" else "死亡"
    printfn $"\nサンプル {i + 1}:"
    printfn $"  特徴量: Pclass={int sample.Pclass}, Sex={sample.Sex}, Age={sample.Age:F0}"
    printfn $"  予測: {predLabel}"
    printfn $"  確信度スコア: {pred.Score:F4}"
)

## まとめ

この Jupyter Notebook での探索により、以下のことが明らかになりました：

- **クラス不均衡の存在** - 死亡 549 人、生存 342 人と偏りあり
- **性別と生存率の強い関係** - 女性の生存率が男性より高い
- **客室クラスと生存率の関係** - 上位クラスほど生存率が高い
- **グループ別欠損値補完の効果** - Pclass と Survived でグループ化して Age を補完
- **モデルの性能** - Accuracy 約 78%, AUC 約 0.84 で良好な分類性能
- **混同行列** - False Negative（見逃し）が課題

### 改善案

- クラス不均衡への対応（SMOTE、重み付けなど）
- より多くの特徴量の利用（Fare, Embarked, SibSp, Parch など）
- 特徴量エンジニアリング（家族サイズ、タイトルなど）
- ハイパーパラメータのチューニング
- アンサンブル学習の適用